<a href="https://colab.research.google.com/github/WhySoSerious404/trust-safety-analytics/blob/main/01-fraud-detection/fraud_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/Project: trust-safety/fraud_oracle.csv')
print(df.shape)

Mounted at /content/drive
(15420, 33)


## Data Exploration

In [3]:
df.head(10)

,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,...,AgeOfVehicle,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange_Claim,NumberOfCars,Year,BasePolicy
0,Dec,5,Wednesday,Honda,Urban,Tuesday,Jan,1,Female,Single,...,3 years,26 to 30,No,No,External,none,1 year,3 to 4,1994,Liability
1,Jan,3,Wednesday,Honda,Urban,Monday,Jan,4,Male,Single,...,6 years,31 to 35,Yes,No,External,none,no change,1 vehicle,1994,Collision
2,Oct,5,Friday,Honda,Urban,Thursday,Nov,2,Male,Married,...,7 years,41 to 50,No,No,External,none,no change,1 vehicle,1994,Collision
3,Jun,2,Saturday,Toyota,Rural,Friday,Jul,1,Male,Married,...,more than 7,51 to 65,Yes,No,External,more than 5,no change,1 vehicle,1994,Liability
4,Jan,5,Monday,Honda,Urban,Tuesday,Feb,2,Female,Single,...,5 years,31 to 35,No,No,External,none,no change,1 vehicle,1994,Collision
5,Oct,4,Friday,Honda,Urban,Wednesday,Nov,1,Male,Single,...,5 years,21 to 25,No,No,External,3 to 5,no change,1 vehicle,1994,Collision
6,Feb,1,Saturday,Honda,Urban,Monday,Feb,3,Male,Married,...,7 years,36 to 40,No,No,External,1 to 2,no change,1 vehicle,1994,Collision
7,Nov,1,Friday,Honda,Urban,Tuesday,Mar,4,Male,Single,...,new,16 to 17,No,No,External,none,no change,1 vehicle,1994,Collision
8,Dec,4,Saturday,Honda,Urban,Wednesday,Dec,5,Male,Single,...,6 years,31 to 35,No,Yes,External,3 to 5,no change,1 vehicle,1994,Collision
9,Apr,3,Tuesday,Ford,Urban,Wednesday,Apr,3,Male,Married,...,more than 7,36 to 40,No,No,External,3 to 5,no change,1 vehicle,1994,All Perils


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15420 entries, 0 to 15419
Data columns (total 33 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Month                 15420 non-null  object
 1   WeekOfMonth           15420 non-null  int64 
 2   DayOfWeek             15420 non-null  object
 3   Make                  15420 non-null  object
 4   AccidentArea          15420 non-null  object
 5   DayOfWeekClaimed      15420 non-null  object
 6   MonthClaimed          15420 non-null  object
 7   WeekOfMonthClaimed    15420 non-null  int64 
 8   Sex                   15420 non-null  object
 9   MaritalStatus         15420 non-null  object
 10  Age                   15420 non-null  int64 
 11  Fault                 15420 non-null  object
 12  PolicyType            15420 non-null  object
 13  VehicleCategory       15420 non-null  object
 14  VehiclePrice          15420 non-null  object
 15  FraudFound_P          15420 non-null

In [5]:
df.isna().sum()

,0
Month,0
WeekOfMonth,0
DayOfWeek,0
Make,0
AccidentArea,0
DayOfWeekClaimed,0
MonthClaimed,0
WeekOfMonthClaimed,0
Sex,0
MaritalStatus,0


In [6]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
WeekOfMonth,15420.0,2.788586,1.287585,1.0,2.00,3.0,4.00,5.0
WeekOfMonthClaimed,15420.0,2.693969,1.259115,1.0,2.00,3.0,4.00,5.0
Age,15420.0,39.855707,13.492377,0.0,31.00,38.0,48.00,80.0
FraudFound_P,15420.0,0.059857,0.237230,0.0,0.00,0.0,0.00,1.0
PolicyNumber,15420.0,7710.500000,4451.514911,1.0,3855.75,7710.5,11565.25,15420.0
RepNumber,15420.0,8.483268,4.599948,1.0,5.00,8.0,12.00,16.0
Deductible,15420.0,407.704280,43.950998,300.0,400.00,400.0,400.00,700.0
DriverRating,15420.0,2.487808,1.119453,1.0,1.00,2.0,3.00,4.0
Year,15420.0,1994.866472,0.803313,1994.0,1994.00,1995.0,1996.00,1996.0


In [7]:
# Count records whose age is zero
zero_age_count = df["Age"].eq(0).sum()

print("Records with age 0:", zero_age_count)
print(f"Percentage with age 0: {zero_age_count / len(df):.2%}")

Records with age 0: 320
Percentage with age 0: 2.08%


In [8]:
# Inspect every observed value and its frequency
for column in ["Deductible", "DriverRating", "Year"]:
    print(f"\n{column}")
    print(df[column].value_counts().sort_index())


Deductible
Deductible
300        8
400    14838
500      263
700      311
Name: count, dtype: int64

DriverRating
DriverRating
1    3944
2    3801
3    3884
4    3791
Name: count, dtype: int64

Year
Year
1994    6142
1995    5195
1996    4083
Name: count, dtype: int64


In [9]:
df.loc[df["Age"].eq(0), "AgeOfPolicyHolder"].value_counts()

,count
AgeOfPolicyHolder,
16 to 17,320


In [10]:
df.loc[
    df["AgeOfPolicyHolder"].eq("16 to 17"),
    "Age"
].value_counts().sort_index()

,count
Age,
0,320


In [22]:
df["Age"].value_counts().sort_index()

,count
Age,
0,320
16,9
17,6
18,48
19,32
...,...
76,42
77,29
78,35


In [23]:
df.groupby("AgeOfPolicyHolder")["Age"].agg(
    ["count", "min", "max"]
)

,count,min,max
AgeOfPolicyHolder,,,
16 to 17,320,0,0
18 to 20,15,16,17
21 to 25,108,18,20
26 to 30,613,21,25
31 to 35,5593,26,35
36 to 40,4043,36,45
41 to 50,2828,46,55
51 to 65,1392,56,65
over 65,508,66,80


In [11]:
df["FraudFound_P"].value_counts().sort_index()

,count
FraudFound_P,
0,14497
1,923


In [12]:
print("Exact duplicate rows:", df.duplicated().sum())

print(
    "Duplicate rows excluding PolicyNumber:",
    df.drop(columns="PolicyNumber").duplicated().sum()
)

print("Unique policy numbers:", df["PolicyNumber"].nunique())

Exact duplicate rows: 0
Duplicate rows excluding PolicyNumber: 0
Unique policy numbers: 15420


In [13]:
df.describe(include="object").T

,count,unique,top,freq
Month,15420,12,Jan,1411
DayOfWeek,15420,7,Monday,2616
Make,15420,19,Pontiac,3837
AccidentArea,15420,2,Urban,13822
DayOfWeekClaimed,15420,8,Monday,3757
MonthClaimed,15420,13,Jan,1446
Sex,15420,2,Male,13000
MaritalStatus,15420,4,Married,10625
Fault,15420,2,Policy Holder,11230
PolicyType,15420,9,Sedan - Collision,5584


In [14]:
df["DayOfWeekClaimed"].value_counts(dropna=False)

,count
DayOfWeekClaimed,
Monday,3757
Tuesday,3375
Wednesday,2951
Thursday,2660
Friday,2497
Saturday,127
Sunday,52
0,1


In [15]:
df["MonthClaimed"].value_counts(dropna=False)

,count
MonthClaimed,
Jan,1446
May,1411
Mar,1348
Oct,1339
Jun,1293
Feb,1287
Nov,1285
Apr,1271
Sep,1242


In [16]:
invalid_day = df["DayOfWeekClaimed"].astype(str).eq("0")
invalid_month = df["MonthClaimed"].astype(str).eq("0")

df.loc[
    invalid_day | invalid_month,
    ["Month", "DayOfWeek", "MonthClaimed",
     "DayOfWeekClaimed", "WeekOfMonthClaimed"]
]

,Month,DayOfWeek,MonthClaimed,DayOfWeekClaimed,WeekOfMonthClaimed
1516,Jul,Monday,0,0,1


In [17]:
for column in df.select_dtypes(include="object").columns:
    print(f"\n{column}:")
    print(df[column].unique())


Month:
['Dec' 'Jan' 'Oct' 'Jun' 'Feb' 'Nov' 'Apr' 'Mar' 'Aug' 'Jul' 'May' 'Sep']

DayOfWeek:
['Wednesday' 'Friday' 'Saturday' 'Monday' 'Tuesday' 'Sunday' 'Thursday']

Make:
['Honda' 'Toyota' 'Ford' 'Mazda' 'Chevrolet' 'Pontiac' 'Accura' 'Dodge'
 'Mercury' 'Jaguar' 'Nisson' 'VW' 'Saab' 'Saturn' 'Porche' 'BMW' 'Mecedes'
 'Ferrari' 'Lexus']

AccidentArea:
['Urban' 'Rural']

DayOfWeekClaimed:
['Tuesday' 'Monday' 'Thursday' 'Friday' 'Wednesday' 'Saturday' 'Sunday'
 '0']

MonthClaimed:
['Jan' 'Nov' 'Jul' 'Feb' 'Mar' 'Dec' 'Apr' 'Aug' 'May' 'Jun' 'Sep' 'Oct'
 '0']

Sex:
['Female' 'Male']

MaritalStatus:
['Single' 'Married' 'Widow' 'Divorced']

Fault:
['Policy Holder' 'Third Party']

PolicyType:
['Sport - Liability' 'Sport - Collision' 'Sedan - Liability'
 'Utility - All Perils' 'Sedan - All Perils' 'Sedan - Collision'
 'Utility - Collision' 'Utility - Liability' 'Sport - All Perils']

VehicleCategory:
['Sport' 'Utility' 'Sedan']

VehiclePrice:
['more than 69000' '20000 to 29000' '30000 to 39

In [18]:
for column in ["Days_Policy_Accident", "Days_Policy_Claim"]:
    print(f"\n{column}")
    print(df[column].value_counts(dropna=False))


Days_Policy_Accident
Days_Policy_Accident
more than 30    15247
none               55
8 to 15            55
15 to 30           49
1 to 7             14
Name: count, dtype: int64

Days_Policy_Claim
Days_Policy_Claim
more than 30    15342
15 to 30           56
8 to 15            21
none                1
Name: count, dtype: int64


In [19]:
df.loc[
    df["Days_Policy_Claim"].eq("none"),
    [
        "MonthClaimed",
        "DayOfWeekClaimed",
        "Days_Policy_Accident",
        "Days_Policy_Claim"
    ]
]

,MonthClaimed,DayOfWeekClaimed,Days_Policy_Accident,Days_Policy_Claim
1516,0,0,more than 30,none


In [20]:
pd.crosstab(df["PolicyType"], df["VehicleCategory"])

VehicleCategory,Sedan,Sport,Utility
PolicyType,,,
Sedan - All Perils,4087,0,0
Sedan - Collision,5584,0,0
Sedan - Liability,0,4987,0
Sport - All Perils,0,22,0
Sport - Collision,0,348,0
Sport - Liability,0,1,0
Utility - All Perils,0,0,340
Utility - Collision,0,0,30
Utility - Liability,0,0,21


In [21]:
pd.crosstab(df["PolicyType"], df["BasePolicy"])

BasePolicy,All Perils,Collision,Liability
PolicyType,,,
Sedan - All Perils,4087,0,0
Sedan - Collision,0,5584,0
Sedan - Liability,0,0,4987
Sport - All Perils,22,0,0
Sport - Collision,0,348,0
Sport - Liability,0,0,1
Utility - All Perils,340,0,0
Utility - Collision,0,30,0
Utility - Liability,0,0,21


### Initial exploration: structure and data quality

#### Dataset and target
- The dataset contains 15,420 records and 33 columns.
- There are 923 fraud-labelled records (5.99%) and 14,497
  non-fraud records (94.01%).

#### Missing values and duplicates
- Initial checks found no missing values.
- One record contains "0" in both MonthClaimed and
  DayOfWeekClaimed, alongside "none" in Days_Policy_Claim.
  The invalid calendar labels will be treated as missing.
  The meaning of "none" remains unverified.
- No duplicate rows were found.
- All 15,420 PolicyNumber values are unique. It will be
  excluded from model inputs as an identifier.

#### Age inconsistencies
- Age is zero in 320 records (2.08%). These records exactly
  match the "16 to 17" AgeOfPolicyHolder category.
- Other AgeOfPolicyHolder labels also disagree with the
  observed Age ranges.
- The cause is undocumented. Original values are preserved.

#### Category distributions and relationships
- Deductible equals 400 in 14,838 records (96.23%).
  Only eight records have a deductible of 300.
- DriverRating values 1–4 have similar counts; the meaning
  and direction of the rating scale remain unverified.
- Text ranges and "none" values need deliberate interpretation
  and encoding.
- BasePolicy consistently matches the coverage portion of
  PolicyType, indicating redundant information.
- All 4,987 "Sedan - Liability" records have VehicleCategory
  recorded as "Sport". This discrepancy is preserved and
  documented.

#### Calendar coverage
- Year ranges from 1994 to 1996, with 6,142, 5,195, and
  4,083 records respectively.

#### Remaining exploration
- Create the training, validation, and test split.
- Compare fraud rates across categories using training data,
  reporting group sizes and fraud counts alongside rates.
- Plot selected category and calendar patterns.
- Add the resulting fraud-pattern findings after those checks.

## Train, validation, and test split